# 06 - Bono: scraping de población por borough

Sacamos la población 2018 de cada borough, con su área y su densidad, de la tabla 2 de *Vital Statistics of New York State 2018* ([NYS Department of Health](https://www.health.ny.gov/statistics/vital_statistics/2018/table02.htm)). La idea es compararla con la que da la encuesta y volver a calcular las tasas. Cada borough es un condado: Bronx = Bronx, Brooklyn = Kings, Manhattan = New York, Queens = Queens y Staten Island = Richmond.

Cuando la página se pide desde un programa responde 403 (lo probamos desde VM1). Por eso la guardamos desde el navegador en `fuentes/nys_vital_statistics_2018_tabla2.html`; no intentamos saltarnos el bloqueo. De ahí en adelante todo es automático.

In [ ]:
import hashlib
import re
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from bs4 import BeautifulSoup
from pyspark.sql import functions as F

from config import RAW, SILVER, sesion
from estilo import BOROUGHS, COLOR_BOROUGH, TINTA_SECUNDARIA, guardar, miles

URL = "https://www.health.ny.gov/statistics/vital_statistics/2018/table02.htm"
COPIA = Path("../fuentes/nys_vital_statistics_2018_tabla2.html")   # guardada desde el navegador
MI2_A_KM2 = 2.589988110336                                    # 1 milla cuadrada = 2,59 km²
CONDADO_A_BOROUGH = {"Bronx": "Bronx", "Kings": "Brooklyn", "New York": "Manhattan",
                     "Queens": "Queens", "Richmond": "Staten Island"}
ANIOS_COMPLETOS = list(range(2019, 2026))

spark = sesion("06_bono_scraping_poblacion")

## 1. Descarga automática

Igual se intenta bajar la página; lo esperado es que responda 403.

In [ ]:
try:
    r = requests.get(URL, timeout=20, headers={"User-Agent": "proyecto-academico-javeriana (python-requests)"})
    print(f"HTTP {r.status_code} al descargar con un programa")
    if r.status_code == 403:
        print("El sitio bloquea clientes automáticos: se usa la copia guardada desde el navegador (no se evade el bloqueo).")
except requests.RequestException as e:
    print("Sin respuesta del sitio:", type(e).__name__, "- se usa la copia guardada desde el navegador.")

## 2. Copia guardada

Se muestran el SHA-256, el título y la fecha de modificación del archivo, y se copia a la capa cruda en HDFS.

In [ ]:
datos = COPIA.read_bytes()
html = datos.decode("utf-8", errors="replace")
sopa = BeautifulSoup(html, "html.parser")

modificada = re.search(r"PAGE LAST MODIFIED ([^-]+-[^-]+-[0-9]{4})", html)
revisada = sopa.find(id="reviseddate")
print("Archivo      :", COPIA.name, f"({len(datos):,} bytes)")
print("SHA-256      :", hashlib.sha256(datos).hexdigest())
print("Título       :", sopa.title.get_text(strip=True))
print("Modificada   :", modificada.group(1).strip() if modificada else "n/d")
print("Revisión     :", revisada.get_text(strip=True) if revisada else "n/d")

# copia a raw en HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark.sparkContext._jsc.hadoopConfiguration())
destino = f"{RAW}/poblacion_condados_2018/{COPIA.name}"
fs.mkdirs(jvm.org.apache.hadoop.fs.Path(f"{RAW}/poblacion_condados_2018"))
fs.copyFromLocalFile(False, True, jvm.org.apache.hadoop.fs.Path(str(COPIA.resolve())), jvm.org.apache.hadoop.fs.Path(destino))
print("Copia en     :", destino)

## 3. Scraping

La página tiene una sola tabla (`class="widetable"`). Las filas con un `<th colspan="5">` son títulos de sección (New York State, New York City, Rest of State) y las que tienen cinco `<td>` son los datos. Los números vienen como texto y con comas.

In [ ]:
tabla = sopa.find("table", class_="widetable")
encabezados = [" ".join(th.get_text(" ", strip=True).split()) for th in tabla.thead.find_all("th")]
print("Encabezados:", encabezados)

def numero(texto):
    return float(texto.replace(",", "").strip())

filas, seccion = [], None
for tr in tabla.tbody.find_all("tr"):
    celdas = tr.find_all("td")
    if not celdas:                                  # fila de título de sección
        seccion = tr.get_text(" ", strip=True)
        continue
    textos = [" ".join(td.get_text(" ", strip=True).split()) for td in celdas]
    filas.append({
        "nombre": textos[0],
        "seccion": seccion,
        "poblacion_2018": int(numero(textos[1])),
        "poblacion_2010": int(numero(textos[2])),
        "area_mi2": numero(textos[3]),
        "densidad_mi2": numero(textos[4]),
    })

AGREGADOS = {"New York State", "New York City", "Rest of State"}
condados = pd.DataFrame(filas)
condados["tipo"] = condados["nombre"].map(lambda n: "agregado" if n in AGREGADOS else "condado")
print(f"{len(condados)} filas leídas: {(condados['tipo'] == 'condado').sum()} condados y {(condados['tipo'] == 'agregado').sum()} totales")
condados.head(10)

## 4. Validación

Se revisa contra los totales que trae la misma tabla: tienen que ser 62 condados, los 5 de la ciudad tienen que sumar el total de NYC, los 62 el total del estado, y la densidad tiene que dar población ÷ área con 2 decimales. En un primer intento comparamos con tolerancia relativa y falló en Hamilton, pero el error estaba en nuestro chequeo y no en la página.

In [ ]:
cond = condados[condados["tipo"] == "condado"]
nyc = cond[cond["seccion"] == "New York City"]
fila = condados.set_index("nombre")
densidad_calculada = cond["poblacion_2018"] / cond["area_mi2"]

validaciones = {
    "62 condados": len(cond) == 62,
    "los 5 condados de NYC están": sorted(nyc["nombre"]) == sorted(CONDADO_A_BOROUGH),
    "NYC = suma de sus 5 condados (2018)": nyc["poblacion_2018"].sum() == fila.loc["New York City", "poblacion_2018"],
    "NYC = suma de sus 5 condados (2010)": nyc["poblacion_2010"].sum() == fila.loc["New York City", "poblacion_2010"],
    "Estado = suma de los 62 condados (2018)": cond["poblacion_2018"].sum() == fila.loc["New York State", "poblacion_2018"],
    "área de NYC = suma de sus condados (±0,05 mi²)": abs(nyc["area_mi2"].sum() - fila.loc["New York City", "area_mi2"]) <= 0.05,
    # la densidad viene con 2 decimales
    "densidad = población ÷ área (a 2 decimales)": ((densidad_calculada - cond["densidad_mi2"]).abs() <= 0.005 + 1e-9).all(),
}
for nombre, ok in validaciones.items():
    print(("OK   " if ok else "FALLA"), nombre)
assert all(validaciones.values()), "La tabla no cuadra: revisar el HTML antes de usar los datos"

## 5. Los cinco boroughs

El área se pasa a km² (1 mi² = 2,59 km²).

In [ ]:
oficial = (nyc.assign(borough=nyc["nombre"].map(CONDADO_A_BOROUGH))
              .set_index("borough").reindex(BOROUGHS))
oficial["area_km2"] = (oficial["area_mi2"] * MI2_A_KM2).round(1)
oficial["densidad_km2"] = (oficial["poblacion_2018"] / oficial["area_km2"]).round(0)
oficial["crecimiento_2010_2018_pct"] = (100 * (oficial["poblacion_2018"] / oficial["poblacion_2010"] - 1)).round(1)
oficial[["nombre", "poblacion_2018", "poblacion_2010", "crecimiento_2010_2018_pct", "area_km2", "densidad_km2"]] \
    .rename(columns={"nombre": "condado"})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(oficial.index, oficial["densidad_km2"], color=[COLOR_BOROUGH[b] for b in oficial.index], width=0.65)
for x, v in enumerate(oficial["densidad_km2"]):
    ax.text(x, v, miles(v), ha="center", va="bottom", fontsize=9, color=TINTA_SECUNDARIA)
ax.yaxis.set_major_formatter(FuncFormatter(miles))
ax.grid(axis="x", visible=False)
ax.set_ylabel("habitantes por km²")
ax.set_title("Densidad de población por borough, 2018", pad=12)
ax.text(0, -0.16, "Fuente: NYS Department of Health, Vital Statistics 2018, Tabla 2 (scraping propio).",
        transform=ax.transAxes, fontsize=8, color=TINTA_SECUNDARIA, va="top")
guardar(fig, "06_densidad_borough")
plt.show()

Manhattan es por mucho el más denso y Staten Island el que menos.

## 6. Población oficial vs encuesta

Hasta ahora la población era la suma de `PWGTP` de la encuesta. Como la encuesta no incluye a la gente en alojamientos colectivos, debería dar menos que la oficial.

In [ ]:
acs = (spark.read.parquet(f"{SILVER}/pobreza").groupBy("borough")
       .agg(F.sum("peso_persona").alias("poblacion_encuesta")).toPandas()
       .set_index("borough").reindex(BOROUGHS))
comparacion = pd.DataFrame({
    "poblacion_oficial_2018": oficial["poblacion_2018"],
    "poblacion_encuesta_2018": acs["poblacion_encuesta"],
})
comparacion["diferencia"] = comparacion["poblacion_oficial_2018"] - comparacion["poblacion_encuesta_2018"]
comparacion["diferencia_pct"] = (100 * comparacion["diferencia"] / comparacion["poblacion_oficial_2018"]).round(1)
comparacion.loc["Total"] = [comparacion[c].sum() for c in ["poblacion_oficial_2018", "poblacion_encuesta_2018", "diferencia"]] + \
    [round(100 * comparacion["diferencia"].sum() / comparacion["poblacion_oficial_2018"].sum(), 1)]
comparacion = comparacion.astype({"poblacion_oficial_2018": "int64", "poblacion_encuesta_2018": "int64", "diferencia": "int64"})
comparacion

En todos los boroughs la encuesta da menos. Puede ser por los alojamientos colectivos, pero eso no lo comprobamos acá.

## 7. Tasas por 100.000 habitantes

Se calculan igual que en el 03 (promedio 2019-2025 y solo registros con borough).

In [ ]:
arrestos = spark.read.parquet(f"{SILVER}/arrestos")
choques = spark.read.parquet(f"{SILVER}/choques")

def conteo_por_borough(df):
    return (df.filter(F.col("anio").isin(ANIOS_COMPLETOS) & F.col("borough").isNotNull())
              .groupBy("borough").count().toPandas().set_index("borough").reindex(BOROUGHS)["count"])

n_arrestos = conteo_por_borough(arrestos)
n_graves = conteo_por_borough(choques.filter(F.col("grave")))

def tasa(conteo, poblacion):
    return (conteo / len(ANIOS_COMPLETOS) / poblacion * 1e5).round(0)

tasas = pd.DataFrame({
    "arrestos_encuesta": tasa(n_arrestos, comparacion.loc[BOROUGHS, "poblacion_encuesta_2018"]),
    "arrestos_oficial": tasa(n_arrestos, oficial["poblacion_2018"]),
    "graves_encuesta": tasa(n_graves, comparacion.loc[BOROUGHS, "poblacion_encuesta_2018"]),
    "graves_oficial": tasa(n_graves, oficial["poblacion_2018"]),
}).astype(int)
tasas["mismo_orden_arrestos"] = tasas["arrestos_encuesta"].rank(ascending=False).eq(tasas["arrestos_oficial"].rank(ascending=False))
tasas["mismo_orden_graves"] = tasas["graves_encuesta"].rank(ascending=False).eq(tasas["graves_oficial"].rank(ascending=False))
tasas

Con la población oficial las tasas bajan un poco. Si todas las columnas `mismo_orden_*` salen `True`, el orden de los boroughs no cambia.

## 8. Guardar resultados

In [ ]:
salida = condados.copy()
salida["borough"] = salida["nombre"].map(CONDADO_A_BOROUGH).where(salida["seccion"] == "New York City")
salida["area_km2"] = salida["area_mi2"] * MI2_A_KM2
salida["fuente"] = "NYS DOH Vital Statistics 2018, Table 2"
esquema = ("nombre string, seccion string, poblacion_2018 long, poblacion_2010 long, area_mi2 double, "
          "densidad_mi2 double, tipo string, borough string, area_km2 double, fuente string")
salida["borough"] = salida["borough"].astype(object).where(salida["borough"].notna(), None)
spark.createDataFrame(salida[[c.split()[0] for c in esquema.split(", ")]], schema=esquema).write.mode("overwrite").parquet(f"{SILVER}/poblacion_condados_2018")
print("Filas en plata:", spark.read.parquet(f"{SILVER}/poblacion_condados_2018").count())

oficial.drop(columns=["seccion", "tipo"]).rename(columns={"nombre": "condado"}).to_csv("../figuras/tabla_06_poblacion_oficial.csv")
comparacion.to_csv("../figuras/tabla_06_poblacion_comparada.csv")
tasas.to_csv("../figuras/tabla_06_tasas_comparadas.csv")
print("Tablas guardadas en figuras/")

## Conclusiones

- La página bloquea los programas (403), así que se usó la copia guardada desde el navegador.
- El scraping recorre el HTML fila por fila y se validó contra los totales de la misma tabla.
- En todos los boroughs la población de la encuesta es menor que la oficial.
- Limitación: solo hay población por condado, no por barrio.